In [78]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [79]:
# !pip install -q "torch==2.5.1" "torchvision==0.20.1" "torchaudio==2.5.1"

In [80]:
# !pip install -q sentence-transformers

In [81]:
# !pip install -q --force-reinstall "torch==2.5.1" "torchvision==0.20.1" "torchaudio==2.5.1"

In [82]:
# !pip install -q sentence-transformers datasets

In [83]:
# !pip uninstall -y torchcodec

In [84]:
# import torch

# print("PyTorch:", torch.__version__)
# print("CUDA:", torch.version.cuda)
# print("GPU:", torch.cuda.get_device_name(0))
# print("Capability:", torch.cuda.get_device_capability(0))
# print("Architectures:", torch.cuda.get_arch_list())

In [85]:
import pandas as pd
import numpy as np
import warnings
import time

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [86]:
# !pip uninstall -y torchcodec

In [87]:
# movies = pd.read_csv("/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/movies.csv")
# print(movies.head())
# print(movies.info())
# print(movies.isnull().sum())

In [88]:
# movies["text"] = (
#     movies["title"].fillna("") + " " +
#     movies["genres"].fillna("") + " " 
# )

In [89]:
# movies["text"].head()

In [90]:
def load_data():
    """
    specify dtypes explicitly to reduce memory usage.
    On a 25M row dataset, this saves ~500MB of RAM.
    """
    print("Loading MovieLens 25M dataset...")
    start = time.time()
    
    # Load ratings 
    ratings = pd.read_csv("/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/ratings.csv",
        dtype={
            'userId': np.int32,
            'movieId': np.int32,
            'rating': np.float32,
            'timestamp': np.int64
        }
    )
    
    # Load movie metadata 
    movies = pd.read_csv("/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/movies.csv")
    
    # Load user-generated tags 
    tags = pd.read_csv("/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/tags.csv",
        dtype={
            'userId': np.int32,
            'movieId': np.int32,
            'timestamp': np.int64
        }
    )
    
    # Convert timestamps to readable dates 
    ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
    ratings['year'] = ratings['datetime'].dt.year
    ratings['month'] = ratings['datetime'].dt.month
    ratings['day_of_week'] = ratings['datetime'].dt.dayofweek
    ratings['hour'] = ratings['datetime'].dt.hour
    
    elapsed = time.time() - start
    
    # Dataset statistics 
    n_users = ratings['userId'].nunique()
    n_items = ratings['movieId'].nunique()
    n_ratings = len(ratings)
    sparsity = 1 - n_ratings / (n_users * n_items)
    
    print(f"Loaded in {elapsed:.1f} seconds")
    print(f"")
    print(f"  Ratings:     {n_ratings:>12,}")
    print(f"  Users:       {n_users:>12,}")
    print(f"  Movies:      {n_items:>12,}")
    print(f"  Tags:        {len(tags):>12,}")
    print(f"  Sparsity:    {sparsity:>12.6f} ({sparsity*100:.2f}%)")
    print(f"")
    print(f"  Date range:  {ratings['datetime'].min().date()} to {ratings['datetime'].max().date()}")
    print(f"  Memory used: {ratings.memory_usage(deep=True).sum() / 1e6:.1f} MB (ratings)")
    
    return ratings, movies, tags


ratings, movies, tags = load_data()

Loading MovieLens 25M dataset...
Loaded in 14.5 seconds

  Ratings:       25,000,095
  Users:            162,541
  Movies:            59,047
  Tags:           1,093,360
  Sparsity:        0.997395 (99.74%)

  Date range:  1995-01-09 to 2019-11-21
  Memory used: 1100.0 MB (ratings)


In [91]:
# ============================================
# STEP 1: Basic preprocessing
# ============================================

print("Starting preprocessing...")

# Make a copy so the original ratings dataframe is preserved
ratings_clean = ratings.copy()

# Remove duplicate ratings
ratings_clean = ratings_clean.drop_duplicates(
    subset=['userId', 'movieId', 'timestamp']
)

# Remove missing values
ratings_clean = ratings_clean.dropna(
    subset=['userId', 'movieId', 'rating', 'timestamp']
)

# Check valid rating range
ratings_clean = ratings_clean[
    (ratings_clean['rating'] >= 0.5) &
    (ratings_clean['rating'] <= 5.0)
]

print(f"Original ratings:       {len(ratings):,}")
print(f"After cleaning:         {len(ratings_clean):,}")

print("\nRating distribution:")
print(ratings_clean['rating'].value_counts().sort_index())


Starting preprocessing...
Original ratings:       25,000,095
After cleaning:         25,000,095

Rating distribution:
rating
0.5     393068
1.0     776815
1.5     399490
2.0    1640868
2.5    1262797
3.0    4896928
3.5    3177318
4.0    6639798
4.5    2200539
5.0    3612474
Name: count, dtype: int64


In [92]:
# ============================================
# STEP 1: Basic preprocessing
# ============================================

print("Starting preprocessing...")
print("-" * 50)

# Make a copy so the original ratings dataframe is preserved
ratings_clean = ratings.copy()

# Remove duplicate ratings, if any
duplicates = ratings_clean.duplicated(
    subset=['userId', 'movieId', 'timestamp']
).sum()

ratings_clean = ratings_clean.drop_duplicates(
    subset=['userId', 'movieId', 'timestamp']
)

# Remove missing values
missing_before = ratings_clean[['userId', 'movieId', 'rating', 'timestamp']].isnull().sum().sum()

ratings_clean = ratings_clean.dropna(
    subset=['userId', 'movieId', 'rating', 'timestamp']
)

# Check valid rating range
ratings_clean = ratings_clean[
    (ratings_clean['rating'] >= 0.5) &
    (ratings_clean['rating'] <= 5.0)
]

print(f"Original ratings:       {len(ratings):,}")
print(f"Duplicate rows removed: {duplicates:,}")
print(f"Missing values removed: {missing_before:,}")
print(f"After cleaning:         {len(ratings_clean):,}")

print("\nRating distribution:")
print(ratings_clean['rating'].value_counts().sort_index())


Starting preprocessing...
--------------------------------------------------
Original ratings:       25,000,095
Duplicate rows removed: 0
Missing values removed: 0
After cleaning:         25,000,095

Rating distribution:
rating
0.5     393068
1.0     776815
1.5     399490
2.0    1640868
2.5    1262797
3.0    4896928
3.5    3177318
4.0    6639798
4.5    2200539
5.0    3612474
Name: count, dtype: int64


In [93]:
# ============================================
# STEP 2: Filter users and movies
# ============================================

MIN_USER_RATINGS = 12
MIN_MOVIE_RATINGS = 64

# Count ratings per user
user_counts = ratings_clean['userId'].value_counts()

# Keep users with at least 5 ratings
valid_users = user_counts[
    user_counts >= MIN_USER_RATINGS
].index

# Count ratings per movie
movie_counts = ratings_clean['movieId'].value_counts()

# Keep movies with at least 5 ratings
valid_movies = movie_counts[
    movie_counts >= MIN_MOVIE_RATINGS
].index

# Filter dataset
ratings_filtered = ratings_clean[
    ratings_clean['userId'].isin(valid_users) &
    ratings_clean['movieId'].isin(valid_movies)
].copy()

print("Filtering results:")
print("-" * 50)
print(f"Ratings: {len(ratings_filtered):,}")
print(f"Users:   {ratings_filtered['userId'].nunique():,}")
print(f"Movies:  {ratings_filtered['movieId'].nunique():,}")


Filtering results:
--------------------------------------------------
Ratings: 24,584,854
Users:   162,540
Movies:  12,103


In [94]:
# ============================================
# PREPARE MOVIE CONTENT
# ============================================

# --- Clean movies ---
movies_clean = movies.copy()

movies_clean["title"] = movies_clean["title"].fillna("").str.strip()
movies_clean["genres"] = movies_clean["genres"].fillna("")

# Convert genres from "Action|Comedy|Drama"
# to "Action Comedy Drama"
movies_clean["genres_text"] = (
    movies_clean["genres"]
    .str.replace("|", " ", regex=False)
)

# --- Process tags ---
tags_clean = tags.copy()

tags_clean = tags_clean.dropna(subset=["movieId", "tag"])

tags_clean["tag"] = (
    tags_clean["tag"]
    .astype(str)
    .str.strip()
)

tags_clean = tags_clean[tags_clean["tag"] != ""]

# Combine all tags belonging to each movie
movie_tags = (
    tags_clean
    .groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.drop_duplicates()))
    .reset_index(name="tag")
)

# --- Merge tags into movies ---
movies_clean = movies_clean.merge(
    movie_tags,
    on="movieId",
    how="left"
)

movies_clean["tag"] = movies_clean["tag"].fillna("")

# --- Create text for Transformer ---
movies_clean["year"] = (
    movies_clean["title"]
    .str.extract(r"\((\d{4})\)")
)

movies_clean["year"] = movies_clean["year"].fillna("unknown")

movies_clean["combined_text"] = (
    "Movie: " + movies_clean["title"] +
    ". Year: " + movies_clean["year"] +
    ". Genres: " + movies_clean["genres_text"] +
    ". Keywords: " + movies_clean["tag"]
)

# Check result
print(movies_clean[
    ["movieId", "title", "genres", "tag", "combined_text"]
].head(10))

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   
5        6                         Heat (1995)   
6        7                      Sabrina (1995)   
7        8                 Tom and Huck (1995)   
8        9                 Sudden Death (1995)   
9       10                    GoldenEye (1995)   

                                        genres  \
0  Adventure|Animation|Children|Comedy|Fantasy   
1                   Adventure|Children|Fantasy   
2                               Comedy|Romance   
3                         Comedy|Drama|Romance   
4                                       Comedy   
5                        Action|Crime|Thriller   
6                               Comedy|Romance   
7                           Adventure|Children   

In [95]:
# ============================================
# STEP 3: Sort chronologically
# ============================================

ratings_filtered = ratings_filtered.sort_values(
    ['userId', 'timestamp']
).reset_index(drop=True)

print("Ratings sorted chronologically.")


Ratings sorted chronologically.


In [96]:
ratings_filtered = ratings_filtered.sort_values(
    ["userId", "timestamp"]
)

test_size = 0.2

train_df = (
    ratings_filtered
    .groupby("userId", group_keys=False)
    .apply(lambda x: x.iloc[:int(len(x) * (1 - test_size))])
)

test_df = (
    ratings_filtered
    .groupby("userId", group_keys=False)
    .apply(lambda x: x.iloc[int(len(x) * (1 - test_size)):])
)

/tmp/ipykernel_58/2043350389.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[:int(len(x) * (1 - test_size))])
/tmp/ipykernel_58/2043350389.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[int(len(x) * (1 - test_size)):])


In [97]:
from itertools import combinations
import random

positive_ratings = train_df[
    train_df["rating"] >= 4
].copy()

MAX_PAIRS_PER_USER = 10
MAX_TOTAL_PAIRS = 400_000

pairs = []

for user_id, group in positive_ratings.groupby("userId"):

    movie_ids = group["movieId"].tolist()

    if len(movie_ids) < 2:
        continue

    user_pairs = list(combinations(movie_ids, 2))

    random.shuffle(user_pairs)

    pairs.extend(user_pairs[:MAX_PAIRS_PER_USER])

    if len(pairs) >= MAX_TOTAL_PAIRS:
        break

pairs = pairs[:MAX_TOTAL_PAIRS]

print("Positive pairs:", len(pairs))

Positive pairs: 400000


In [98]:
movie_text = dict(
    zip(
        movies_clean["movieId"],
        movies_clean["combined_text"]
    )
)

In [99]:
training_pairs = []

for movie_a, movie_b in pairs:

    if movie_a not in movie_text:
        continue

    if movie_b not in movie_text:
        continue

    training_pairs.append({
        "anchor": movie_text[movie_a],
        "positive": movie_text[movie_b]
    })

print("Training pairs:", len(training_pairs))

Training pairs: 400000


In [100]:
from datasets import Dataset

train_dataset = Dataset.from_list(training_pairs)

train_dataset

Dataset({
    features: ['anchor', 'positive'],
    num_rows: 400000
})

In [101]:
from sentence_transformers.losses import MultipleNegativesRankingLoss



In [102]:
# ============================================
# STEP 5: Create relevance labels
# ============================================

RELEVANCE_THRESHOLD = 4.0

train_df['relevant'] = (
    train_df['rating'] >= RELEVANCE_THRESHOLD
).astype(np.int8)

test_df['relevant'] = (
    test_df['rating'] >= RELEVANCE_THRESHOLD
).astype(np.int8)

print("Relevance distribution")
print("-" * 50)

print("Training:")
print(train_df['relevant'].value_counts())
print(
    f"Positive: {train_df['relevant'].mean()*100:.2f}%"
)

print("\nTesting:")
print(test_df['relevant'].value_counts())
print(
    f"Positive: {test_df['relevant'].mean()*100:.2f}%"
)

Relevance distribution
--------------------------------------------------
Training:
relevant
1    9974850
0    9628966
Name: count, dtype: int64
Positive: 50.88%

Testing:
relevant
0    2626431
1    2354607
Name: count, dtype: int64
Positive: 47.27%


In [103]:
# ============================================
# STEP 3: Generate movie embeddings
# ============================================

from sentence_transformers import SentenceTransformer

# Load pretrained Transformer
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)
loss = MultipleNegativesRankingLoss(model)
print("Device:", model.device)

# Generate embeddings
embeddings = model.encode(
    movies_clean["combined_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device="cuda"

)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda:0


Batches:   0%|          | 0/976 [00:00<?, ?it/s]

Embedding shape: (62423, 384)


In [104]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments
)

args = SentenceTransformerTrainingArguments(
    output_dir="movie-minilm-finetuned",

    num_train_epochs=2,

    per_device_train_batch_size=32,

    learning_rate=2e-5,

    warmup_steps=500,

    fp16=True,

    logging_steps=100,

    save_strategy="epoch"
)

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss
)

trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,4.837543
200,4.020946
300,3.962101
400,3.928440
500,3.911001
600,3.908641


In [ ]:
model.save_pretrained(
    "movie-minilm-finetuned"
)

In [ ]:
model = SentenceTransformer(
    "movie-minilm-finetuned"
)

In [ ]:
fine_tuned_embeddings = model.encode(
    movies_clean["combined_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device="cuda"
)

print(fine_tuned_embeddings.shape)

In [ ]:
np.save(
    "movie_embeddings_finetuned.npy",
    fine_tuned_embeddings
)

In [ ]:
embeddings = fine_tuned_embeddings

In [ ]:
# ============================================
# STEP 4: Movie similarity search
# ============================================

from sklearn.metrics.pairwise import cosine_similarity

def get_similar_movies(movie_title, top_n=10):

    # Find movie
    matches = movies_clean[
    movies_clean["title"].str.contains(
        str(movie_title),
        case=False,
        na=False,
        regex=False
        )
    ]

    if matches.empty:
        print(f"Movie not found: {movie_title}")
        return pd.DataFrame(
            columns=["movieId", "content_score"]
        )

    # Use first matching movie
    movie_index = matches.index[0]

    # Get embedding
    query_embedding = embeddings[movie_index].reshape(1, -1)

    # Calculate similarity
    similarities = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    # Get highest similarity scores
    similar_indices = similarities.argsort()[::-1]

    # Remove the movie itself
    similar_indices = [
        i for i in similar_indices
        if i != movie_index
    ]

    # Select top N
    similar_indices = similar_indices[:top_n]

    recommendations = movies_clean.iloc[similar_indices][
        ["movieId", "title", "genres"]
    ].copy()

    recommendations["content_score"] = similarities[similar_indices]

    return recommendations

In [ ]:
get_similar_movies("Toy Story", top_n=10)

In [ ]:
!pip install faiss-cpu

In [ ]:
# ============================================
# STEP 5: Build FAISS similarity index
# ============================================

import faiss
import numpy as np

# Make sure embeddings are float32

embeddings_f32 = fine_tuned_embeddings.astype("float32")

embedding_dim = embeddings_f32.shape[1]

index = faiss.IndexFlatIP(
    embedding_dim
)

index.add(embeddings_f32)

print(
    "Indexed movies:",
    index.ntotal
)

In [ ]:
def recommend_movies(movie_title, top_n=10):

    # Find movie
    matches = movies_clean[
        movies_clean["title"].str.contains(
            movie_title,
            case=False,
            na=False
        )
    ]

    if len(matches) == 0:
        print("Movie not found.")
        return

    movie_index = matches.index[0]

    # Get embedding
    query_vector = embeddings_f32[movie_index].reshape(1, -1)

    # Search FAISS index
    scores, indices = index.search(
        query_vector,
        top_n + 1
    )

    # Remove the query movie itself
    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == movie_index:
            continue

        results.append({
            "movieId": movies_clean.iloc[idx]["movieId"],
            "title": movies_clean.iloc[idx]["title"],
            "genres": movies_clean.iloc[idx]["genres"],
            "similarity": score
        })

        if len(results) == top_n:
            break

    return pd.DataFrame(results)

In [ ]:
recommend_movies("Toy Story", top_n=10)

In [ ]:
# Rating >= 4 means the user liked the movie
ratings_filtered["relevant"] = (
    ratings_filtered["rating"] >= 4
)

In [ ]:
# ============================================
# STEP 6: Temporal train/test split
# ============================================

ratings_sorted = ratings_filtered.sort_values(
    ["userId", "timestamp"]
)

# Last rating of each user → test set
test = (
    ratings_sorted
    .groupby("userId")
    .tail(5)
)

# Everything before the last rating → training set
train = ratings_sorted.drop(test.index)

print("Training ratings:", len(train))
print("Test ratings:", len(test))

In [ ]:
def hit_rate_at_k(user_id, k=10):

    # Movies the user liked in training
    user_train = train[
        (train["userId"] == user_id) &
        (train["rating"] >= 4)
    ]

    if len(user_train) == 0:
        return 0

    # Actual test movie
    user_test = test[
        (test["userId"] == user_id)
    ]

    if len(user_test) == 0:
        return 0

    test_movie = user_test.iloc[0]["movieId"]

    # Pick a movie the user liked previously
    seed_movie = user_train.iloc[-1]["movieId"]

    # Get its index
    matches = movies_clean.index[
        movies_clean["movieId"] == seed_movie
    ]

    if len(matches) == 0:
        return 0

    seed_index = matches[0]

    query_vector = embeddings_f32[
        seed_index
    ].reshape(1, -1)

    scores, indices = index.search(
        query_vector,
        k + 1
    )

    recommended_movie_ids = [
        movies_clean.iloc[idx]["movieId"]
        for idx in indices[0]
        if idx != seed_index
    ][:k]

    return int(test_movie in recommended_movie_ids)

In [ ]:
users = test["userId"].unique()

hits = []

for user_id in users[:1000]:
    hits.append(
        hit_rate_at_k(user_id, k=10)
    )

hit_rate = np.mean(hits)

print(f"Hit Rate@10: {hit_rate:.4f}")

In [ ]:
# ============================================
# STEP 7: COLLABORATIVE FILTERING
# ============================================

from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

print("Building collaborative filtering matrix...")

# Use TRAINING ratings only
cf_data = train[["userId", "movieId", "rating"]].copy()

# Create integer indices
user_ids = cf_data["userId"].unique()
movie_ids = cf_data["movieId"].unique()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}

idx_to_movie = {
    idx: movie_id
    for movie_id, idx in movie_to_idx.items()
}

# Convert IDs to matrix indices
cf_data["user_idx"] = cf_data["userId"].map(user_to_idx)
cf_data["movie_idx"] = cf_data["movieId"].map(movie_to_idx)

# Build Movie × User rating matrix
movie_user_matrix = csr_matrix(
    (
        cf_data["rating"].values,
        (
            cf_data["movie_idx"].values,
            cf_data["user_idx"].values
        )
    ),
    shape=(
        len(movie_ids),
        len(user_ids)
    ),
    dtype=np.float32
)

print("Movie-user matrix shape:", movie_user_matrix.shape)
print("Non-zero ratings:", movie_user_matrix.nnz)

In [ ]:
# ============================================
# Mean-center ratings for collaborative filtering
# ============================================

movie_means = np.asarray(
    movie_user_matrix.sum(axis=1)
).ravel()

movie_counts = np.diff(
    movie_user_matrix.indptr
)

movie_means = (
    movie_means /
    np.maximum(movie_counts, 1)
)

# Convert sparse matrix to COO format
movie_user_coo = movie_user_matrix.tocoo()

centered_data = (
    movie_user_coo.data -
    movie_means[movie_user_coo.row]
)

centered_matrix = csr_matrix(
    (
        centered_data,
        (
            movie_user_coo.row,
            movie_user_coo.col
        )
    ),
    shape=movie_user_matrix.shape,
    dtype=np.float32
)

print("Centered matrix created.")

In [ ]:
# Normalize each movie vector
centered_matrix = normalize(
    centered_matrix,
    norm="l2",
    axis=1
)

print("Collaborative matrix normalized.")

In [ ]:
# ============================================
# Collaborative similarity
# ============================================

def get_collaborative_similar_movies(
    movie_id,
    top_n=50
):
    
    if movie_id not in movie_to_idx:
        return pd.DataFrame(
            columns=["movieId", "collab_score"]
        )
    
    movie_idx = movie_to_idx[movie_id]
    
    query_vector = centered_matrix[
        movie_idx
    ]
    
    # Sparse matrix multiplication
    similarities = centered_matrix @ query_vector.T
    
    similarities = similarities.toarray().ravel()
    
    # Highest similarities
    similar_indices = np.argsort(
        similarities
    )[::-1]
    
    results = []
    
    for idx in similar_indices:
        
        # Skip itself
        if idx == movie_idx:
            continue
        
        score = similarities[idx]
        
        results.append({
            "movieId": idx_to_movie[idx],
            "collab_score": float(score)
        })
        
        if len(results) >= top_n:
            break
    
    return pd.DataFrame(results)

In [ ]:
toy_story_id = 1

get_collaborative_similar_movies(
    toy_story_id,
    top_n=10
)

In [ ]:
# ============================================
# STEP 8: CONTENT + COLLABORATIVE HYBRID
# ============================================

CONTENT_WEIGHT = 0.7
COLLAB_WEIGHT = 0.3


def hybrid_recommend(
    movie_title,
    top_n=10,
    content_weight=CONTENT_WEIGHT,
    collab_weight=COLLAB_WEIGHT
):
    
    # ----------------------------------------
    # 1. Find input movie
    # ----------------------------------------
    
    matches = movies_clean[
        movies_clean["title"].str.contains(
            movie_title,
            case=False,
            na=False
        )
    ]
    
    if len(matches) == 0:
        print("Movie not found.")
        return pd.DataFrame()
    
    movie_index = matches.index[0]
    
    input_movie = movies_clean.loc[
        movie_index,
        "title"
    ]
    
    input_movie_id = movies_clean.loc[
        movie_index,
        "movieId"
    ]
    
    print("Input movie:", input_movie)
    
    
    # ----------------------------------------
    # 2. CONTENT RECOMMENDATIONS
    # ----------------------------------------
    
    query_vector = embeddings_f32[
        movie_index
    ].reshape(1, -1)
    
    scores, indices = index.search(
        query_vector,
        min(top_n + 100, len(movies_clean))
    )
    
    content_results = []
    
    for score, idx in zip(
        scores[0],
        indices[0]
    ):
        
        if idx == movie_index:
            continue
        
        movie_id = movies_clean.iloc[
            idx
        ]["movieId"]
        
        content_results.append({
            "movieId": movie_id,
            "content_score": float(score)
        })
    
    content_df = pd.DataFrame(
        content_results
    )
    
    
    # ----------------------------------------
    # 3. COLLABORATIVE RECOMMENDATIONS
    # ----------------------------------------
    
    collab_df = get_collaborative_similar_movies(
        input_movie_id,
        top_n=100
    )
    
    
    # ----------------------------------------
    # 4. Combine content + collaborative
    # ----------------------------------------
    
    recommendations = pd.merge(
        content_df,
        collab_df,
        on="movieId",
        how="inner"
    )
    
    
    # ----------------------------------------
    # 5. Handle missing scores
    # ----------------------------------------
    
    recommendations["content_score"] = (
        recommendations["content_score"]
        .fillna(0)
    )
    
    recommendations["collab_score"] = (
        recommendations["collab_score"]
        .fillna(0)
    )
    
    
    # ----------------------------------------
    # 6. Normalize content score
    # ----------------------------------------
    
    content_min = (
        recommendations["content_score"].min()
    )
    
    content_max = (
        recommendations["content_score"].max()
    )
    
    recommendations["content_score_norm"] = (
        recommendations["content_score"]
        - content_min
    ) / (
        content_max
        - content_min
        + 1e-8
    )
    
    
    # ----------------------------------------
    # 7. Normalize collaborative score
    # ----------------------------------------
    
    collab_min = (
        recommendations["collab_score"].min()
    )
    
    collab_max = (
        recommendations["collab_score"].max()
    )
    
    recommendations["collab_score_norm"] = (
        recommendations["collab_score"]
        - collab_min
    ) / (
        collab_max
        - collab_min
        + 1e-8
    )
    
    
    # ----------------------------------------
    # 8. Calculate hybrid score
    # ----------------------------------------
    
    recommendations["hybrid_score"] = (
        content_weight *
        recommendations["content_score_norm"]
        +
        collab_weight *
        recommendations["collab_score_norm"]
    )
    
    
    # ----------------------------------------
    # 9. Sort recommendations
    # ----------------------------------------
    
    recommendations = (
        recommendations
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(top_n)
    )
    
    
    # ----------------------------------------
    # 10. Add movie information
    # ----------------------------------------
    
    recommendations = recommendations.merge(
        movies_clean[
            [
                "movieId",
                "title",
                "genres"
            ]
        ],
        on="movieId",
        how="left"
    )
    
    
    # ----------------------------------------
    # 11. Final output
    # ----------------------------------------
    
    return recommendations[
        [
            "movieId",
            "title",
            "genres",
            "content_score",
            "collab_score",
            "content_score_norm",
            "collab_score_norm",
            "hybrid_score"
        ]
    ]

In [ ]:
recommendations = hybrid_recommend(
    "Toy Story",
    top_n=10
)

recommendations

In [ ]:
def recommend_for_user(
    user_id,
    top_n=10,
    content_weight=0.7,
    collab_weight=0.3
):
    
    # -----------------------------------------
    # 1. Check if user exists
    # -----------------------------------------
    if user_id not in train["userId"].values:
        return pd.DataFrame(
            columns=[
                "movieId",
                "title",
                "genres",
                "content_score",
                "collab_score",
                "hybrid_score"
            ]
        )
    
    # Movies already rated by this user
    user_movies = set(
        train.loc[
            train["userId"] == user_id,
            "movieId"
        ]
    )
    
    if not user_movies:
        return pd.DataFrame()
    
    # -----------------------------------------
    # 2. Generate candidates
    # -----------------------------------------
    content_candidates = []
    collab_candidates = []
    
    for movie_id in user_movies:
        
        # =====================================
        # CONTENT-BASED
        # =====================================
        
        # Find title corresponding to movieId
        movie_row = movies_clean[
            movies_clean["movieId"] == movie_id
        ]
        
        if not movie_row.empty:
            
            movie_title = movie_row.iloc[0]["title"]
            
            # get_similar_movies expects TITLE
            content_results = get_similar_movies(
                movie_title,
                top_n=50
            )
            
            if not content_results.empty:
                content_candidates.append(
                    content_results
                )
        
        # =====================================
        # COLLABORATIVE FILTERING
        # =====================================
        
        collab_results = get_collaborative_similar_movies(
            movie_id,
            top_n=50
        )
        
        if not collab_results.empty:
            collab_candidates.append(
                collab_results
            )
    
    # -----------------------------------------
    # 3. Combine content candidates
    # -----------------------------------------
    if content_candidates:
        
        content_df = pd.concat(
            content_candidates,
            ignore_index=True
        )
        
        content_df = (
            content_df
            .groupby("movieId", as_index=False)["content_score"]
            .max()
        )
        
    else:
        content_df = pd.DataFrame(
            columns=["movieId", "content_score"]
        )
    
    # -----------------------------------------
    # 4. Combine collaborative candidates
    # -----------------------------------------
    if collab_candidates:
        
        collab_df = pd.concat(
            collab_candidates,
            ignore_index=True
        )
        
        collab_df = (
            collab_df
            .groupby("movieId", as_index=False)["collab_score"]
            .max()
        )
        
    else:
        collab_df = pd.DataFrame(
            columns=["movieId", "collab_score"]
        )
    
    # -----------------------------------------
    # 5. Merge content + collaborative
    # -----------------------------------------
    recommendations = pd.merge(
        content_df,
        collab_df,
        on="movieId",
        how="inner"
    )
    
    # Remove movies already watched/rated
    recommendations = recommendations[
        ~recommendations["movieId"].isin(user_movies)
    ].copy()
    
    if recommendations.empty:
        return pd.DataFrame()
    
    # -----------------------------------------
    # 6. Normalize scores
    # -----------------------------------------
    def min_max_normalize(series):
        
        min_val = series.min()
        max_val = series.max()
        
        if max_val == min_val:
            return pd.Series(
                1.0,
                index=series.index
            )
        
        return (
            (series - min_val) /
            (max_val - min_val)
        )
    
    recommendations["content_score_norm"] = (
        min_max_normalize(
            recommendations["content_score"]
        )
    )
    
    recommendations["collab_score_norm"] = (
        min_max_normalize(
            recommendations["collab_score"]
        )
    )
    
    # -----------------------------------------
    # 7. Calculate hybrid score
    # -----------------------------------------
    recommendations["hybrid_score"] = (
        content_weight *
        recommendations["content_score_norm"]
        +
        collab_weight *
        recommendations["collab_score_norm"]
    )
    
    # -----------------------------------------
    # 8. Sort and select top N
    # -----------------------------------------
    recommendations = (
        recommendations
        .sort_values(
            "hybrid_score",
            ascending=False
        )
        .head(top_n)
    )
    
    # -----------------------------------------
    # 9. Add title and genres
    # -----------------------------------------
    recommendations = recommendations.merge(
        movies_clean[
            ["movieId", "title", "genres"]
        ],
        on="movieId",
        how="left"
    )
    
    return recommendations[
        [
            "movieId",
            "title",
            "genres",
            "content_score",
            "collab_score",
            "hybrid_score"
        ]
    ].reset_index(drop=True)

In [ ]:
user_id = test["userId"].iloc[0]

recommendations = recommend_for_user(
    user_id,
    top_n=10,
    content_weight=0.7,
    collab_weight=0.3
)

recommendations

In [ ]:
def evaluate_hit_rate(
    user_ids,
    k=10,
    content_weight=0.7,
    collab_weight=0.3
):
    
    hits = 0
    evaluated = 0
    
    for user_id in user_ids:
        
        if user_id not in test_relevant:
            continue
        
        recommendations = recommend_for_user(
            user_id,
            top_n=k,
            content_weight=content_weight,
            collab_weight=collab_weight
        )
        
        if recommendations.empty:
            continue
        
        recommended_ids = set(
            recommendations["movieId"]
        )
        
        actual_ids = test_relevant[user_id]
        
        if recommended_ids & actual_ids:
            hits += 1
        
        evaluated += 1
    
    if evaluated == 0:
        return 0
    
    return hits / evaluated

In [ ]:
# # Create the set of relevant movies for each user
# # A rating >= 4 is considered a positive/relevant interaction

# test_relevant = (
#     test[test["rating"] >= 4]
#     .groupby("userId")["movieId"]
#     .apply(set)
#     .to_dict()
# )

# print("Users with relevant test movies:", len(test_relevant))

In [ ]:
# evaluation_users = list(
#     test_relevant.keys()
# )[:1000]

# hit_rate = evaluate_hit_rate(
#     evaluation_users,
#     k=10
# )

# print(f"Hit Rate@10: {hit_rate:.4f}")

In [ ]:
# for content_weight in [1.0, 0.9, 0.8, 0.7, 0.5]:
    
#     collab_weight = 1 - content_weight
    
#     score = evaluate_hit_rate(
#         evaluation_users,
#         k=10,
#         content_weight=content_weight,
#         collab_weight=collab_weight
#     )
    
#     print(
#         f"Content={content_weight:.1f}, "
#         f"Collaborative={collab_weight:.1f}, "
#         f"Hit Rate@10={score:.4f}"
#     )

In [ ]:
# ============================================
# EVALUATION
# ============================================

# Create the set of relevant movies for each user
# A rating >= 4 is considered a positive/relevant interaction

test_relevant = (
    test[test["rating"] >= 4]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print(
    "Users with relevant test movies:",
    len(test_relevant)
)


# Evaluate on first 1000 users
evaluation_users = list(
    test_relevant.keys()
)[:100]


# Test different Content / Collaborative weights
for content_weight in [1.0, 0.9, 0.8, 0.7, 0.5]:
    
    collab_weight = 1 - content_weight
    
    score = evaluate_hit_rate(
        evaluation_users,
        k=10,
        content_weight=content_weight,
        collab_weight=collab_weight
    )
    
    print(
        f"Content={content_weight:.1f}, "
        f"Collaborative={collab_weight:.1f}, "
        f"Hit Rate@10={score:.4f}"
    )

In [ ]:
def evaluate_precision(
    user_ids,
    k=10,
    content_weight=0.7,
    collab_weight=0.3
):
    
    precisions = []

    for user_id in user_ids:

        if user_id not in test_relevant:
            continue

        recommendations = recommend_for_user(
            user_id,
            top_n=k,
            content_weight=content_weight,
            collab_weight=collab_weight
        )

        if recommendations.empty:
            continue

        recommended_ids = set(
            recommendations["movieId"]
        )

        actual_ids = test_relevant[user_id]

        relevant_recommended = (
            recommended_ids & actual_ids
        )

        precision = (
            len(relevant_recommended) / k
        )

        precisions.append(precision)

    if not precisions:
        return 0

    return np.mean(precisions)

In [ ]:
import time

start = time.time()

precision_10 = evaluate_precision(
    evaluation_users[:10],
    k=10,
    content_weight=0.7,
    collab_weight=0.3
)

elapsed = time.time() - start

print(f"Precision@10: {precision_10:.4f}")
print(f"10 users: {elapsed:.2f} seconds")
print(f"Estimated 1000 users: {elapsed * 100 / 60:.2f} minutes")

In [ ]:
def evaluate_recall(
    user_ids,
    k=10,
    content_weight=0.7,
    collab_weight=0.3
):
    
    recalls = []

    for user_id in user_ids:

        if user_id not in test_relevant:
            continue

        recommendations = recommend_for_user(
            user_id,
            top_n=k,
            content_weight=content_weight,
            collab_weight=collab_weight
        )

        if recommendations.empty:
            continue

        recommended_ids = set(
            recommendations["movieId"]
        )

        actual_ids = test_relevant[user_id]

        relevant_recommended = (
            recommended_ids & actual_ids
        )

        recall = (
            len(relevant_recommended)
            / len(actual_ids)
        )

        recalls.append(recall)

    if not recalls:
        return 0

    return np.mean(recalls)

In [ ]:
from sklearn.metrics import ndcg_score

In [ ]:
def evaluate_ndcg(
    user_ids,
    k=10,
    content_weight=0.7,
    collab_weight=0.3
):
    
    ndcgs = []

    for user_id in user_ids:

        if user_id not in test_relevant:
            continue

        recommendations = recommend_for_user(
            user_id,
            top_n=k,
            content_weight=content_weight,
            collab_weight=collab_weight
        )

        if recommendations.empty:
            continue

        recommended_ids = (
            recommendations["movieId"].tolist()
        )

        actual_ids = test_relevant[user_id]

        # Relevance of each recommended movie
        relevance = [
            1 if movie_id in actual_ids else 0
            for movie_id in recommended_ids
        ]

        if not any(relevance):
            ndcgs.append(0)
            continue

        # Scores represent ranking position
        scores = [
            k - i
            for i in range(len(relevance))
        ]

        ndcg = ndcg_score(
            [relevance],
            [scores],
            k=k
        )

        ndcgs.append(ndcg)

    if not ndcgs:
        return 0

    return np.mean(ndcgs)

In [ ]:
def evaluate_ndcg(
    user_ids,
    k=10,
    content_weight=0.7,
    collab_weight=0.3
):
    
    ndcgs = []

    for user_id in user_ids:

        if user_id not in test_relevant:
            continue

        recommendations = recommend_for_user(
            user_id,
            top_n=k,
            content_weight=content_weight,
            collab_weight=collab_weight
        )

        if recommendations.empty:
            continue

        recommended_ids = recommendations["movieId"].tolist()

        actual_ids = test_relevant[user_id]

        relevance = [
            1 if movie_id in actual_ids else 0
            for movie_id in recommended_ids
        ]

        if not any(relevance):
            ndcgs.append(0)
            continue

        # Scores represent the recommendation ranking
        scores = [
            k - i
            for i in range(len(relevance))
        ]

        ndcg = ndcg_score(
            [relevance],
            [scores],
            k=k
        )

        ndcgs.append(ndcg)

    if not ndcgs:
        return 0

    return np.mean(ndcgs)

In [ ]:
K = 10

precision = evaluate_precision(
    evaluation_users,
    k=K,
    content_weight=0.7,
    collab_weight=0.3
)

recall = evaluate_recall(
    evaluation_users,
    k=K,
    content_weight=0.7,
    collab_weight=0.3
)

ndcg = evaluate_ndcg(
    evaluation_users,
    k=K,
    content_weight=0.7,
    collab_weight=0.3
)

hit_rate = evaluate_hit_rate(
    evaluation_users,
    k=K,
    content_weight=0.7,
    collab_weight=0.3
)

print(f"Precision@{K}: {precision:.4f}")
print(f"Recall@{K}:    {recall:.4f}")
print(f"NDCG@{K}:      {ndcg:.4f}")
print(f"Hit Rate@{K}:  {hit_rate:.4f}")